In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
train_sample_path = "train_sample.csv"
test_sample_path = "test_sample.csv"

In [7]:
train_sample = pd.read_csv(train_sample_path)
train_sample.head()

FileNotFoundError: [Errno 2] No such file or directory: '.../train_sample.csv'

In [ ]:
test_sample = pd.read_csv(test_sample_path)
test_sample.head(3)

In [ ]:
train_sample.info()

In [ ]:
train_sample.describe()

In [ ]:
train_sample.isnull().sum()

In [ ]:
train_sample.duplicated().sum()

In [ ]:
train_sample.fillna(train_sample.mean(), inplace=True)
train_sample.isnull().sum()

In [ ]:
test_sample.fillna(test_sample.mean(), inplace=True)
test_sample.isnull().sum()

In [ ]:
train_sample.drop_duplicates(inplace=True)
train_sample.duplicated().sum()

(opsional) drop column

In [ ]:
cols_to_drop = ['id', 'date']
train_sample = train_sample.drop(columns=cols_to_drop)
train_sample.head(2)

In [ ]:
test_sample = test_sample.drop(columns=cols_to_drop)
test_sample.head(2)

train val split

In [ ]:
X_train, y_train, X_val, y_val = train_test_split(train_sample.drop(columns=['target']), train_sample['target'], test_size=0.2, random_state=38)

one-hot encoding

In [ ]:
# ohe = OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore')
# X_train_encoded = ohe.fit_transform(X_train)
# X_val_encoded = ohe.transform(X_val)
# X_test_encoded = ohe.transform(test_sample)
# X_train_encoded.head(3)

In [ ]:
from sklearn.model_selection import cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

categorical_cols = ['Gender', 'Parental Education', 'Lunch Type']
numerical_cols = ['Hours Studied', 'Previous Scores']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
)

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(train_sample)

pd.DataFrame(y_pred).to_csv('data/submission.csv')